# G1 Academy Bonus - Task 8: high-level arm gestures, teach/repeat, and low-level pose interpolation

## Introduction
This task covers `notes.txt` section 6 in full: native high-level arm gestures through `G1ArmActionClient`, then a controller-ownership-safe path onto `rt/arm_sdk` (`release_arms`/`engage_arms`), and finally two general-purpose low-level pose helpers - `save_current_ll_pose` / `interpolate_to_ll_pose` - built up into `teach`/`repeat` for recording and replaying new multi-waypoint arm sequences.

In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowState_

_factory_config = None
def ensure_channel_factory(domain_id, interface):
    global _factory_config
    config = (int(domain_id), str(interface))
    if _factory_config is None:
        ChannelFactoryInitialize(*config)
        _factory_config = config
    elif _factory_config != config:
        raise RuntimeError(f"ChannelFactory already initialized as {_factory_config}; restart kernel for {config}.")
    return _factory_config

ensure_channel_factory(0, "eth0")

class Latest:
    def __init__(self, topic, message_type, queue_len=10):
        self.message = None
        self.timestamp = 0.0
        self.subscriber = ChannelSubscriber(topic, message_type)
        self.subscriber.Init(self._callback, queue_len)
    def _callback(self, message):
        self.message = message
        self.timestamp = time.time()
    def fresh(self, max_age_s=0.5):
        return self.message is not None and time.time() - self.timestamp <= max_age_s

lowstate_sub = Latest("rt/lowstate", LowState_)

## Task 1 - High-level gestures via `G1ArmActionClient`
`G1ArmActionClient.ExecuteAction(action_id)` triggers a documented, firmware-baked gesture. These must never run concurrently with a low-level `rt/arm_sdk` command stream - finish or release one before starting the other.

In [ ]:
from unitree_sdk2py.g1.arm.g1_arm_action_client import G1ArmActionClient

HL_ARM_ACTIONS = {"release arm": 99, "clap": 17, "face wave": 25, "high wave": 26, "shake hand": 27, "hug": 19}

arm_action_client = G1ArmActionClient(); arm_action_client.SetTimeout(10.0); arm_action_client.Init()

def exec_arm_action(name, release_after_s=None):
    code = int(arm_action_client.ExecuteAction(HL_ARM_ACTIONS[name]))
    if release_after_s is not None:
        time.sleep(release_after_s)
        return int(arm_action_client.ExecuteAction(HL_ARM_ACTIONS["release arm"]))
    return code

def clap():
    return exec_arm_action("clap")

def face_wave():
    return exec_arm_action("face wave")

# clap()
# face_wave()

## Task 2 - Upper-body pose read + `rt/arm_sdk` writer
Every low-level helper below reads the current upper-body pose from `rt/lowstate` and writes a complete `LowCmd_` frame to `rt/arm_sdk`, including the weight byte at `motor_cmd[29]` that arbitrates between the default controller and this publisher.

In [ ]:
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__LowCmd_
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowCmd_
from unitree_sdk2py.utils.crc import CRC

WAIST_JOINTS = (12, 13, 14)
UPPER_BODY_JOINTS = list(WAIST_JOINTS) + list(range(15, 22)) + list(range(22, 29))
_crc = CRC()
arm_sdk_pub = ChannelPublisher("rt/arm_sdk", LowCmd_); arm_sdk_pub.Init()

def current_upper_body_pose(timeout_s=3.0):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if lowstate_sub.message is not None:
            return {j: float(lowstate_sub.message.motor_state[j].q) for j in UPPER_BODY_JOINTS}
        time.sleep(0.02)
    raise TimeoutError("No fresh rt/lowstate.")

def write_arm_sdk_pose(targets, weight=1.0, kp=30.0, kd=1.5, waist_kp=480.0, waist_kd=12.0):
    msg = unitree_hg_msg_dds__LowCmd_(); msg.mode_pr = 0; msg.mode_machine = 0
    msg.motor_cmd[29].q = max(0.0, min(1.0, float(weight)))
    for joint, q in targets.items():
        cmd = msg.motor_cmd[int(joint)]
        cmd.mode = 1; cmd.q = float(q); cmd.dq = 0.0; cmd.tau = 0.0
        cmd.kp = waist_kp if int(joint) in WAIST_JOINTS else kp
        cmd.kd = waist_kd if int(joint) in WAIST_JOINTS else kd
    msg.crc = _crc.Crc(msg)
    arm_sdk_pub.Write(msg)

## Task 3 - `release_arms` / `engage_arms`: safe controller handoff
Ramp the `arm_sdk` weight smoothly (an ease curve, not a step) from 1 to 0 to hand control back to the default controller, or 0 to 1 to take it - this is what every low-level helper below must run before/after it owns `rt/arm_sdk`.

In [ ]:
def release_arms(steps=150, rate_hz=50.0):
    pose = current_upper_body_pose()
    for i in range(steps + 1):
        ratio = i / steps
        fade = ratio * ratio * (3 - 2 * ratio)
        weight = 1.0 - fade
        write_arm_sdk_pose(pose, weight=weight, kp=30.0 * weight, kd=1.5 * weight, waist_kp=480.0 * weight, waist_kd=12.0 * weight)
        time.sleep(1.0 / rate_hz)
    return {"final_weight": 0.0}

def engage_arms(steps=50, rate_hz=50.0):
    pose = current_upper_body_pose()
    for i in range(steps + 1):
        weight = i / steps
        write_arm_sdk_pose(pose, weight=weight)
        time.sleep(1.0 / rate_hz)
    return {"final_weight": 1.0}

# engage_arms()
# release_arms()

## Task 4 - `save_current_ll_pose(name)` / `interpolate_to_ll_pose(name_or_pose, ...)`
Two general-purpose helpers: capture the current pose under a name (persisted to JSON so it survives a kernel restart), and smoothly interpolate from wherever the arm currently is to a saved (or literal) pose using the same ease curve as `release_arms`/`engage_arms`.

In [ ]:
import json
from pathlib import Path

_ll_pose_store_path = Path("ll_poses.json")
def _load_ll_poses():
    return json.loads(_ll_pose_store_path.read_text()) if _ll_pose_store_path.exists() else {}
def _save_ll_poses(poses):
    _ll_pose_store_path.write_text(json.dumps(poses, indent=2))
_ll_poses = _load_ll_poses()

def save_current_ll_pose(name):
    pose = current_upper_body_pose()
    _ll_poses[str(name)] = pose
    _save_ll_poses(_ll_poses)
    return pose

def interpolate_to_ll_pose(name_or_pose, duration_s=4.0, steps=150):
    target = _ll_poses[name_or_pose] if isinstance(name_or_pose, str) else name_or_pose
    target = {int(j): float(q) for j, q in target.items()}
    start = current_upper_body_pose()
    for step in range(1, steps + 1):
        ratio = step / steps
        smooth = ratio * ratio * (3 - 2 * ratio)
        frame = {j: start[j] + (target[j] - start[j]) * smooth for j in target}
        write_arm_sdk_pose(frame)
        time.sleep(duration_s / steps)
    return {"target": target, "steps": steps}

# save_current_ll_pose("extended_right")
# interpolate_to_ll_pose("extended_right", duration_s=4.0)

## Task 5 - `teach(sequence_name)` / `repeat(sequence_name)`: recording new sequences
`teach` appends the current pose as the next waypoint of a named, persisted sequence - call it repeatedly (moving the arm by hand in damp mode, or via `interpolate_to_ll_pose` to intermediate poses, between calls) to build up a multi-waypoint motion. `repeat` plays every captured waypoint back in order using `interpolate_to_ll_pose`, so playback is exactly as smooth as any other low-level move here.

In [ ]:
_sequences_path = Path("arm_sequences.json")
def _load_sequences():
    return json.loads(_sequences_path.read_text()) if _sequences_path.exists() else {}
def _save_sequences(seqs):
    _sequences_path.write_text(json.dumps(seqs, indent=2))
_sequences = _load_sequences()

def teach(sequence_name, reset=False):
    if reset or sequence_name not in _sequences:
        _sequences[sequence_name] = []
    _sequences[sequence_name].append(current_upper_body_pose())
    _save_sequences(_sequences)
    return len(_sequences[sequence_name])

def repeat(sequence_name, waypoint_duration_s=3.0, steps_per_waypoint=100):
    for waypoint in _sequences[sequence_name]:
        interpolate_to_ll_pose(waypoint, duration_s=waypoint_duration_s, steps=steps_per_waypoint)
    return {"sequence": sequence_name, "waypoints": len(_sequences[sequence_name])}

# teach("wave_sequence", reset=True)  # capture waypoint 1
# teach("wave_sequence")              # capture waypoint 2, etc.
# repeat("wave_sequence")

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.